In [1]:
# ==========================================
# Find Spark
# ==========================================
import findspark
findspark.init()

In [2]:
# ==========================================
# Imports
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# ==========================================
# Create Spark Session
# ==========================================

spark = SparkSession.builder \
    .appName("API Data Pipeline") \
    .getOrCreate()

In [ ]:
# ==========================================
# Reading Silver layer 
# ==========================================

silver_df = spark.read.parquet(
    "D:/api-data-pipeline/data/silver/posts"
)
silver_df.show(5, truncate=False)
print("Silver Record Count:", silver_df.count())

+---+------+--------------------------------------------------------------------------+------------+--------------+
|id |userId|title                                                                     |title_length|title_category|
+---+------+--------------------------------------------------------------------------+------------+--------------+
|1  |1     |sunt aut facere repellat provident occaecati excepturi optio reprehenderit|74          |Long          |
|2  |1     |qui est esse                                                              |12          |Short         |
|3  |1     |ea molestias quasi exercitationem repellat qui ipsa sit aut               |59          |Long          |
|4  |1     |eum et est occaecati                                                      |20          |Short         |
|5  |1     |nesciunt quas odio                                                        |18          |Short         |
+---+------+------------------------------------------------------------

In [14]:
# ==========================================
# Average of length per User 
# ==========================================

gold_layer=silver_df.groupBy("userId").agg(F.count("id").alias("total_count"), F.avg("title_length").alias("average_length"))
gold_layer.show(50)  
gold_layer.printSchema()
gold_layer.show(50)

+------+-----------+--------------+
|userId|total_count|average_length|
+------+-----------+--------------+
|     7|         10|          37.4|
|     6|         10|          42.1|
|     9|         10|          42.3|
|     5|         10|          48.4|
|     1|         10|          33.8|
|    10|         10|          35.4|
|     3|         10|          37.1|
|     8|         10|          43.6|
|     2|         10|          40.0|
|     4|         10|          35.1|
+------+-----------+--------------+

root
 |-- userId: long (nullable = true)
 |-- total_count: long (nullable = false)
 |-- average_length: double (nullable = true)

+------+-----------+--------------+
|userId|total_count|average_length|
+------+-----------+--------------+
|     7|         10|          37.4|
|     6|         10|          42.1|
|     9|         10|          42.3|
|     5|         10|          48.4|
|     1|         10|          33.8|
|    10|         10|          35.4|
|     3|         10|          37.1|
|    

In [15]:
print("Silver records:", silver_df.count())
print("Gold users:", gold_layer.count())
print("Gold total posts:", gold_layer.agg(F.sum("total_count")).collect()[0][0])

Silver records: 100
Gold users: 10
Gold total posts: 100


In [16]:
gold_path = "D:/api-data-pipeline/data/gold/posts_summary"

gold_layer.write \
    .mode("overwrite") \
    .parquet(gold_path)

In [18]:
gold_df = spark.read.parquet(gold_path)
gold_df.show()

+------+-----------+--------------+
|userId|total_count|average_length|
+------+-----------+--------------+
|     7|         10|          37.4|
|     6|         10|          42.1|
|     9|         10|          42.3|
|     5|         10|          48.4|
|     1|         10|          33.8|
|    10|         10|          35.4|
|     3|         10|          37.1|
|     8|         10|          43.6|
|     2|         10|          40.0|
|     4|         10|          35.1|
+------+-----------+--------------+

